# ComfyUI on Colab (free T4) — Img2Img + Upscaler

Run every cell top to bottom. Runtime > Change runtime type > **T4 GPU** first.

This sets up ComfyUI, downloads a base SD1.5 checkpoint + a RealESRGAN upscale model, launches the server, and exposes it publicly via a free Cloudflare Tunnel (no account needed). The final cell prints the public URL you'll paste into the GlitrAI backend's `COMFYUI_BASE_URL` setting.

**Note:** the `trycloudflare.com` URL is ephemeral — it changes every time you rerun the tunnel cell or restart the runtime. You'll need to update `COMFYUI_BASE_URL` each time.

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
!git clone https://github.com/comfyanonymous/ComfyUI
%cd ComfyUI
!pip install -r requirements.txt -q

## Download models
Base SD1.5 checkpoint (official successor repo to runwayml/stable-diffusion-v1-5) and a RealESRGAN 4x upscale model.

In [ ]:
%cd /content/ComfyUI
!wget -q --show-progress -O models/checkpoints/v1-5-pruned-emaonly.safetensors \
  https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors
!wget -q --show-progress -O models/upscale_models/RealESRGAN_x4plus.pth \
  https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth
print('Done. If either download failed (0 bytes), open the URL manually in a browser,\n'
      'download the file, and upload it into the matching models/ subfolder via the Colab file browser.')

## (Optional) Use a different / better product-photography checkpoint
The vanilla SD1.5 checkpoint above is a safe, guaranteed-to-download default. For noticeably better product/lifestyle photorealism, download a realistic checkpoint's `.safetensors` file from Civitai or Hugging Face yourself, upload it into `models/checkpoints/`, and set that filename as `ckpt_name` in `workflow_api.json` (node `"1"`).

In [ ]:
import subprocess, threading, time

comfyui_process = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
    cwd="/content/ComfyUI",
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

def _stream(proc):
    for line in proc.stdout:
        print(line, end="")

threading.Thread(target=_stream, args=(comfyui_process,), daemon=True).start()
print("Starting ComfyUI server, waiting ~20s for it to boot...")
time.sleep(20)
print("If you see 'Starting server' / 'To see the GUI go to' above, it's ready.")

In [ ]:
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared

In [ ]:
import subprocess, threading, re

tunnel_process = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://localhost:8188"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

url_found = threading.Event()
public_url = {"value": None}
pattern = re.compile(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com")

def _watch(proc):
    for line in proc.stdout:
        print(line, end="")
        match = pattern.search(line)
        if match and not url_found.is_set():
            public_url["value"] = match.group(0)
            url_found.set()

threading.Thread(target=_watch, args=(tunnel_process,), daemon=True).start()
url_found.wait(timeout=60)
print("\n\n=== PUBLIC COMFYUI URL (paste as COMFYUI_BASE_URL) ===")
print(public_url["value"])

## Sanity check from inside Colab
Loads `workflow_api.json` (upload it into `/content/ComfyUI/` via the file browser, or paste its contents into a `%%writefile` cell), swaps in a test prompt, and queues it directly against the local server to confirm the workflow itself is valid before wiring up the external backend.

In [ ]:
import json, urllib.request

with open("/content/ComfyUI/workflow_api.json") as f:
    graph = json.load(f)

graph["2"]["inputs"]["text"] = (
    "Florentine wooden salad bowl on a sunlit kitchen table, fresh salad inside, "
    "soft natural light, shallow depth of field, commercial product photography"
)

req = urllib.request.Request(
    "http://localhost:8188/prompt",
    data=json.dumps({"prompt": graph}).encode(),
    headers={"Content-Type": "application/json"},
)
print(urllib.request.urlopen(req).read().decode())
print("\nNote: node 4 (LoadImage) needs an actual uploaded reference image filename \n"
      "to run end to end \u2014 upload one via the ComfyUI web UI at the public URL above \n"
      "first (drag an image into a LoadImage node), or use the backend integration \n"
      "which uploads it via the /upload/image API automatically.")